# Семинар 05. Генераторы


## Цели

После семинара вы сможете:

- различать итерируемый объект, итератор и генератор;
- объяснять ленивое выполнение и однократный обход;
- создавать конечные и бесконечные генераторы с помощью `yield` и `yield from`;
- понимать, когда генератор экономит память, а когда лишь скрывает её расход.

## Перед началом

Повторите функции, циклы, списки, области видимости, `iter()` и `next()`.


## Итерируемый объект, итератор и генератор

Эти понятия связаны, но обозначают разные вещи.

| Понятие | Что обязано уметь | Примеры |
|---|---|---|
| Итерируемый объект (`iterable`) | Возвращать итератор по вызову `iter(obj)` | `list`, `str`, `range`, файл |
| Итератор (`iterator`) | Возвращать следующее значение по `next(obj)` и после исчерпания выбрасывать `StopIteration` | результат `iter(list)`, файл |
| Генератор (`generator`) | Быть итератором, созданным генераторной функцией или выражением | `countdown(3)`, `(x * x for x in data)` |

Контейнер обычно можно обойти заново: каждый вызов `iter(container)` создаёт новый итератор. Это похоже на один и тот же документ с несколькими независимыми курсорами: документ хранит данные, а каждый курсор — свою текущую позицию. Итератор и есть такой движущийся курсор, рассчитанный на один проход. Для любого корректного итератора `iter(iterator) is iterator`.

Цикл `for item in source` сначала получает итератор через `iter(source)`, затем вызывает `next()` до `StopIteration`. Сам цикл обрабатывает это исключение и завершает обход, поэтому вручную ловить `StopIteration` в обычном коде почти никогда не нужно.

```python
numbers = [10, 20]
iterator = iter(numbers)

assert iter(iterator) is iterator
assert next(iterator) == 10
assert next(iterator) == 20
# Следующий next(iterator) выбросит StopIteration.
```

## Как работает генераторная функция

Если в теле функции есть `yield`, Python создаёт генераторную функцию. Её вызов не выполняет тело функции, а сразу возвращает объект-генератор. Выполнение начинается по `next()` и приостанавливается, когда доходит до `yield`: локальные переменные и текущая позиция в коде сохраняются. Следующий `next()` продолжает работу с того же места, а `return` завершает генератор.

```python
from collections.abc import Iterator

def countdown(start: int) -> Iterator[int]:
    print("Начали")
    while start > 0:
        yield start
        start -= 1

counter = countdown(3)  # Пока ничего не напечатано.
assert next(counter) == 3
assert list(counter) == [2, 1]  # Оставшиеся значения.
assert list(counter) == []      # Генератор уже исчерпан.
```

Исчерпанный генератор нельзя вернуть в начальное состояние. Если нужен повторный обход, создайте новый генератор или заранее сохраните значения в коллекцию.

## Ленивость и память

Генератор вычисляет очередное значение только тогда, когда его запросили. Вместо заранее заполненного склада получается производство по требованию: потребитель берёт одно значение, после чего вычисляется следующее. Это позволяет строить конвейеры и работать с потоками, которые велики или вообще бесконечны.

Генераторное выражение `(transform(x) for x in source)` работает лениво, а списковое включение `[transform(x) for x in source]` сразу создаёт весь список.

Ленивость не гарантирует постоянный расход памяти. Генератор может удерживать исходные объекты или накапливать состояние. В примере ниже список `primes` растёт без ограничения: значения выдаются по одному, но все найденные простые числа продолжают храниться. Генератор не создаёт готовый бесконечный результат, однако память этого алгоритма всё равно растёт.

Для построчной обработки файла отдельный генератор вообще не обязателен: файловый объект уже поддерживает итерацию.

```python
def process_file(filename: str) -> None:
    with open(filename, encoding="utf-8") as file:
        for line in file:
            process_line(line)
```

`readlines()` здесь был бы ошибкой: он сначала загрузил бы весь файл. Вызов `list(infinite_generator())` не завершится вообще и будет занимать память, пока процесс не остановят или он не упадёт из-за её нехватки.


In [ ]:
from collections.abc import Iterator
from itertools import islice


def prime_generator() -> Iterator[int]:
    # Локальное состояние сохраняется между вызовами next().
    primes = []
    candidate = 2

    while True:
        is_prime = all(
            candidate % prime != 0
            for prime in primes
            if prime * prime <= candidate
        )
        if is_prime:
            primes.append(candidate)
            yield candidate

        candidate = 3 if candidate == 2 else candidate + 2


first_fourteen_primes = list(islice(prime_generator(), 14))
print(first_fourteen_primes)


## `yield from`, `send()` и завершение

`yield from iterable` последовательно передаёт наружу значения другого итерируемого объекта. Для такого делегирования запись точнее и короче ручного цикла с `yield`.

```python
from collections.abc import Iterable, Iterator

def chain(*sources: Iterable[int]) -> Iterator[int]:
    for source in sources:
        yield from source

assert list(chain([1, 2], range(3, 5))) == [1, 2, 3, 4]
```

Обычно генератор только отдаёт значения. Метод `send(value)` позволяет передать значение внутрь при возобновлении: оно становится результатом приостановленного выражения `yield`. До первого `yield` генератор ещё не готов принять значение, поэтому его запускают через `next()` или `send(None)`. Передать на этом этапе любое другое значение нельзя. Это двусторонний протокол, а не обязательный способ писать генераторы; без реальной необходимости он только усложняет API.


In [ ]:
from collections.abc import Generator


def running_total() -> Generator[int, int, None]:
    total = 0
    while True:
        value = yield total
        total += value


accumulator = running_total()
assert next(accumulator) == 0
assert accumulator.send(5) == 5
assert accumulator.send(7) == 12
accumulator.close()


`return` или достижение конца функции завершает генератор: следующий запрос значения приводит к `StopIteration`, который цикл `for` обрабатывает сам. Метод `close()` передаёт в остановленный генератор `GeneratorExit` и прекращает его работу. Если генератор владеет ресурсом, освобождайте ресурс через `with` или `try/finally`; полагаться на момент запуска сборщика мусора нельзя.

## Самопроверка

1. Почему список является итерируемым объектом, но не итератором?
2. Когда начинает выполняться тело генераторной функции?
3. Что произойдёт при повторном вызове `list()` для уже исчерпанного генератора?
4. Почему генератор простых чисел выше всё равно расходует всё больше памяти?
5. Зачем перед первым `send(value)` вызывают `next()`?

## Итоги

- Итерируемый объект создаёт итератор; итератор выдаёт значения и хранит позицию; генератор — один из видов итератора.
- `yield` приостанавливает выполнение с сохранением состояния, а `return` завершает генератор.
- Генераторы однопроходные. Повторный обход требует нового генератора или сохранённой коллекции.
- Ленивое выполнение помогает не создавать весь результат заранее, но не ограничивает память автоматически.
- `yield from` делегирует обход, `send()` организует двусторонний обмен, `close()` завершает генератор.

Документация Python: [глоссарий по итераторам и генераторам](https://docs.python.org/3/glossary.html), [выражение `yield`](https://docs.python.org/3/reference/expressions.html#yield-expressions).


## Задание 1. Генератор названий

Напишите генератор названий мебели IKEA или сообщений УВБ-76 на русском языке. Каждое значение должно состоять из 3–8 букв.

```python
from collections.abc import Iterator

def name_generator() -> Iterator[str]:
    ...
```

**Критерии проверки:** функция возвращает генератор, а не готовый список; последовательные вызовы `next()` дают строки допустимой длины и алфавита.

Могут пригодиться `itertools.product`, `random.shuffle` и `random.choices`.


## Задание 2. Ограничения на сочетания букв

Усложните предыдущий генератор:

- в слове есть хотя бы одна гласная;
- подряд идут не более двух гласных;
- подряд идут не более трёх согласных.

**Критерии проверки:** все ограничения выполняются одновременно; генератор способен выдать не менее 100 значений без завершения; проверки вынесены в отдельные функции.
